# Kolokvijum I — teorija iza rešenja

Ovaj notebook prati **samo rešenje**, deo po deo. Za svaki mehanizam objašnjava zašto je tu,
koju rečenicu iz zadatka rešava, i daje jednu sliku koja ga čini očiglednim.

## Jedna stvar koju vredi primetiti odmah

Zadatak nije slučajno o **glavnoj knjizi**. Knjigovodstvo je funkcionalno već pet vekova:

- unos se **nikad ne briše i ne prepravlja** — greška se ispravlja **novim** unosom (storniranjem),
- saldo se **ne čuva** kao broj koji neko ažurira — dobija se **sabiranjem** unosa,
- stara stanja knjige moraju ostati **proverljiva** unazad.

Sve što zadatak traži — nemutiranje, izvedeno stanje, istorija, storniranje — **već je pravilo struke**.
Ne prevodiš programerski hir na knjigovodstvo; prevodiš knjigovodstvo u kod koji ga ne kvari.

## Rečenica iz zadatka → mehanizam

| Šta zadatak traži | Čime je rešeno | Odeljak |
| --- | --- | --- |
| „redni broj je **privatni statički** atribut" | IIFE oko konstruktora | 1 |
| „vrednosti mogu biti nerealizovana, realizovana, stornirana" | spisak u IIFE + `includes` | 1 |
| „naziv, matični broj, pib i transakcije" | polja na `this` + geter | 2 |
| „metode **ne smeju da mutiraju** original" | geter vraća kopiju + `new` | 2 i 4 |
| „izvedeni atribut **stanje**" | `Object.defineProperty` sa `get` | 3 |
| „rezultat je **nova instanca**" | `new GlavnaKnjiga(...)` | 4 |
| „istorija izmena kao **referenca na prethodnu**" | peti argument = `this` | 4 i 5 |
| „kriterijumi se primenjuju **redom kojim su navedeni**" | `reduce` nad filterima | 6 |
| „opoziv **ne sme da menja** nijednu knjigu ni transakciju" | kopija umesto izmene | 7 |

Sve ćelije se izvršavaju redom i grade rešenje kakvo predaješ.

---
# 1 · `Transakcija` — matičar i matična knjiga

## Slika

U opštini sedi **matičar**. Pored njega stoji **jedna knjiga** u kojoj je zapisan sledeći slobodan broj.
Kad izdaje izvod, upiše taj broj u izvod i uveća broj u knjizi.

- Knjiga je **jedna za celu opštinu** — ne po izvodu. To je **statički**.
- Knjiga je **iza šaltera** — ne možeš je uzeti ni prelistati. To je **privatno**.
- Broj koji dobiješ **piše na tvom izvodu** i tvoj je zauvek.

Prevedeno: brojač je jedan za ceo tip, nedostupan spolja, a `redniBroj` je polje pojedinačne transakcije.

```js
const Transakcija = (() => {
  let brojac = 0;                    // knjiga iza šaltera
  ...
  return function (...) {
    brojac++;
    const redniBroj = brojac;        // broj upisan u ovaj izvod
```

Zagrade `(() => { ... })()` znače da se ta funkcija izvrši **odmah i samo jednom**. Zbog toga postoji
**jedna** knjiga. Da je `let brojac` bio unutar konstruktora, svaki `new` bi otvorio novu knjigu i
svi bi dobili broj 1.

## Spisak dozvoljenih — i zašto provera ide PRE brojača

Matičar ima i spisak dozvoljenih unosa. Ako tražiš nešto van spiska, odbija te — **i ne troši broj na tebe**.
Sledeći na redu dobija broj koji je tebi bio namenjen.

Zato u kodu provera stoji **iznad** `brojac++`. Da je ispod, odbijeni pokušaji bi pravili rupe u numeraciji.

In [ ]:
const Transakcija = (() => {
  let brojac = 0;
  const STATUSI = ["nerealizovana", "realizovana", "stornirana"];
  const TIPOVI = ["na teret", "u korist"];

  return function (brojRacuna, opis, status, datum, tip, iznos) {
    if (!STATUSI.includes(status)) throw new Error("Nedozvoljen status: " + status);
    if (!TIPOVI.includes(tip)) throw new Error("Nedozvoljen tip: " + tip);

    brojac++;
    const redniBroj = brojac;

    this.brojRacuna = brojRacuna;
    this.opis = opis;
    this.status = status;
    this.datum = datum;
    this.tip = tip;
    this.iznos = iznos;

    Object.defineProperty(this, "redniBroj", {
      get() { return redniBroj; },
      enumerable: true,
    });
  };
})();

const t1 = new Transakcija("265-1", "Uplata", "realizovana", "2026-08-01", "u korist", 120000);
const t2 = new Transakcija("265-1", "Zakup", "realizovana", "2026-08-03", "na teret", 45000);

console.log("jedna knjiga za sve:", t1.redniBroj, t2.redniBroj);
console.log("knjiga je iza saltera:", typeof brojac);

// odbijen zahtev ne trosi broj
try { new Transakcija("265-1", "X", "izmisljen", "2026-08-04", "u korist", 1); }
catch (g) { console.log("\nodbijeno:", g.message); }
try { new Transakcija("265-1", "X", "realizovana", "2026-08-04", "u dobit", 1); }
catch (g) { console.log("odbijeno:", g.message); }

const t3 = new Transakcija("265-2", "Nabavka", "nerealizovana", "2026-09-01", "na teret", 90000);
console.log("sledeci dobija:", t3.redniBroj, "— dva odbijena nisu potrosila broj");

// broj na izvodu se ne prepravlja
// (u <script> upis tiho ne uspe; u strogom rezimu baca, pa je u try)
try { t1.redniBroj = 999; } catch (g) { console.log("upis odbijen:", g.constructor.name); }
console.log("posle pokusaja prepravke:", t1.redniBroj, "— geter bez setera");

---
# 2 · `GlavnaKnjiga` — sef i šalter

## Slika

Spisak stoji **u sefu**. Do njega se ne dolazi. Na šalteru dobiješ **fotokopiju spiska**.

Precrtaj na njoj šta hoćeš, dopiši redove, pocepaj je — **spisak u sefu ostaje isti**.

Ali pazi šta je tu kopirano: **spisak**, ne i dokumenti na koje spisak upućuje. Svaki red je i dalje
**uput na isti dokument u arhivi**. Ako odeš u arhivu i pišeš po samom dokumentu, to je pravi dokument —
i videće ga svako ko na njega upućuje. Vratićemo se na to u odeljku 7, jer je baš tu razlog zašto
`opoziv` mora da pravi kopije.

```js
const GlavnaKnjiga = function (naziv, maticniBroj, pib, transakcije = [], prethodna = null) {
  // `transakcije` je PARAMETAR — nije nigde na this, pa je u sefu

  Object.defineProperty(this, "transakcije", {
    get() { return [...transakcije]; },        // šalter izdaje fotokopiju
    enumerable: true,
  });
```

Primeti šta je javno, a šta nije:

| | Gde stoji | Kako se dolazi |
| --- | --- | --- |
| `naziv`, `maticniBroj`, `pib` | `this.naziv = naziv` | direktno, i sme |
| `prethodna` | `this.prethodna` | direktno — `istorija` mora da prošeta lanac |
| `transakcije` | **samo parametar** | geter, i to kao **kopija** |

Bez tog `[...transakcije]` u geteru, `knjiga.transakcije.push(...)` bi menjao samu knjigu spolja —
i cela priča o nemutiranju bi pala, iako metode rade ispravno.

In [ ]:
const GlavnaKnjiga = function (naziv, maticniBroj, pib, transakcije = [], prethodna = null) {
  this.naziv = naziv;
  this.maticniBroj = maticniBroj;
  this.pib = pib;
  this.prethodna = prethodna;

  Object.defineProperty(this, "transakcije", {
    get() { return [...transakcije]; },
    enumerable: true,
  });

  Object.defineProperty(this, "stanje", {
    get() {
      return transakcije
        .filter((t) => t.status === "realizovana")
        .reduce((acc, t) => (t.tip === "u korist" ? acc + t.iznos : acc - t.iznos), 0);
    },
    enumerable: true,
  });

  this.dodajTransakciju = function (transakcija) {
    return new GlavnaKnjiga(naziv, maticniBroj, pib, [...transakcije, transakcija], this);
  };

  this.ukloniTransakciju = function (redniBroj) {
    return new GlavnaKnjiga(naziv, maticniBroj, pib,
      transakcije.filter((t) => t.redniBroj !== redniBroj), this);
  };
};

const k0 = new GlavnaKnjiga("Merkur", "214", "108");
const k1 = k0.dodajTransakciju(t1);
const k2 = k1.dodajTransakciju(t2);
const k3 = k2.dodajTransakciju(t3);

console.log("sa saltera:", k3.naziv, k3.maticniBroj, k3.pib);
console.log("fotokopija:", k3.transakcije.map((t) => t.opis));

// cepanje fotokopije ne dira sef
k3.transakcije.push(t1);
console.log("\nposle push-a na kopiju:", k3.transakcije.length, "— SPISAK je zasticen");
console.log("svaki put nova fotokopija:", k3.transakcije === k3.transakcije);

// ali stavke na spisku su isti dokumenti
k3.transakcije[0].opis = "PRECRTANO";
console.log("\nposle upisa u stavku:", k3.transakcije[0].opis, "| t1.opis:", t1.opis);
console.log("   STAVKA nije zasticena — spisak je kopiran, dokumenti nisu");
k3.transakcije[0].opis = "Uplata";     // vracamo kako je bilo
console.log("   vraceno:", t1.opis);

---
# 3 · `stanje` — saldo se ne čuva, nego se sabira

## Slika

Zamisli da knjigovođa pored knjige drži **cedulju sa saldom** i ažurira je pri svakom unosu.
Šta se dešava ako jednom zaboravi? Cedulja i knjiga se **raziđu** — a onda ni jednom ni drugom
ne možeš da veruješ.

Zato se saldo **ne čuva**. On je **pitanje**, ne podatak: *saberi realizovane u korist, oduzmi
realizovane na teret*. Odgovor se računa svaki put iznova, pa se ne može raziđi sa sadržajem.

To je smisao reči „**izvedeni** atribut" u zadatku.

```js
Object.defineProperty(this, "stanje", {
  get() { ... },        // get, ne vrednost
});
```

Pošto je `get`, čita se **bez zagrada** — `knjiga.stanje`, ne `knjiga.stanje()`. Spolja izgleda kao
obično polje, iako iza njega stoji računanje.

## Zašto `filter` pa `reduce`

Tekst kaže „razlika suma **realizovanih** transakcija u korist i na teret". Dve odluke:

- **koje se broje** → `filter` izbaci sve što nije realizovano
- **sa kojim znakom** → `reduce` sabira ili oduzima po tipu

In [ ]:
console.log("stanje:", k3.stanje, "= 120000 - 45000");
console.log("nerealizovana od 90000 se ne racuna");
console.log("prazna knjiga:", k0.stanje, "— nema uskladistene vrednosti koju bi vratila");

console.log("\ncita se bez zagrada:", typeof k3.stanje, "| kao polje:", "stanje" in k3);

// nema cedulje koja bi zastarela: svaka verzija racuna svoje
console.log("\nsvaka verzija svoje stanje:", [k0.stanje, k1.stanje, k2.stanje, k3.stanje]);

---
# 4 · `dodajTransakciju` i `ukloniTransakciju` — knjiga se ne gumica

## Slika

U knjigovodstvu **ne postoji gumica**. Kad treba nešto promeniti, ne prepravlja se stari list —
otvara se **novi**, a na njemu piše od kog je nastao.

Zato metode ne menjaju knjigu nego prave novu:

```js
this.dodajTransakciju = function (transakcija) {
  return new GlavnaKnjiga(naziv, maticniBroj, pib, [...transakcije, transakcija], this);
  //                                                ↑ nov niz              ↑ od koga je nastala
};
```

Dva detalja u tom jednom redu:

**`[...transakcije, transakcija]`** pravi **nov** niz. Da je `transakcije.push(...)`, stara knjiga bi
se promenila — jer bi delile isti niz.

**`this` kao peti argument** je odgovor na „istorija izmena u vidu reference na prethodnu glavnu
knjigu". Nova knjiga u polju `prethodna` drži **onu od koje je nastala**. Istorija nije poseban
dnevnik koji se vodi — **nastaje sama**, kao posledica načina gradnje.

## Šta se time dobija

Stara verzija ostaje **potpuno upotrebljiva**. Ne „približno" — ista je kao pre, sa svojim
transakcijama i svojim saldom. To je ono što reviziju čini mogućom.

In [ ]:
const dodata = k3.dodajTransakciju(
  new Transakcija("265-1", "Kamata", "realizovana", "2026-09-05", "u korist", 4000));
const uklonjena = k3.ukloniTransakciju(t2.redniBroj);

console.log("nov list:", dodata.transakcije.map((t) => t.opis), "| stanje:", dodata.stanje);
console.log("drugi list:", uklonjena.transakcije.map((t) => t.opis), "| stanje:", uklonjena.stanje);
console.log("stari list netaknut:", k3.transakcije.map((t) => t.opis), "| stanje:", k3.stanje);

console.log("\nnove instance:", k3 !== dodata, k3 !== uklonjena);
console.log("na svakoj pise od koga je nastala:", dodata.prethodna === k3, uklonjena.prethodna === k3);

// transakcije se DELE, ne kopiraju — nov je samo niz
console.log("\nista transakcija u obe knjige:", k3.transakcije[0] === dodata.transakcije[0]);
console.log("   zato nova verzija kosta koliko i kopija spiska, ne koliko cela knjiga");

---
# 5 · `istorija` — lanac unazad

## Slika

Svaki list nosi napomenu „prethodni list: …". Ako uzmeš poslednji i ideš unazad, prošetaćeš celu
istoriju. Prvi list nema tu napomenu — po tome znaš da si stigao do početka.

```js
const istorija = function (knjiga) {
  return knjiga === null ? [] : [...istorija(knjiga.prethodna), knjiga];
};
```

Pročitaj naglas: *istorija ničega je prazan niz; istorija knjige je istorija njene prethodnice,
pa zatim ona sama.*

To je **definicija**, a ne postupak. Nema brojača, nema petlje, nema niza koji se usput menja.

## Zašto izlazi poređano od najstarije

Zato što `istorija(knjiga.prethodna)` stoji **ispred** `knjiga` u nizu. Da je obrnuto —
`[knjiga, ...istorija(knjiga.prethodna)]` — dobio bi obrnut redosled, bez ijedne druge izmene.

Redosled je bitan jer `opoziv` prima **indeks u istoriji**, pa `0` mora biti najstarija.

In [ ]:
const istorija = function (knjiga) {
  return knjiga === null ? [] : [...istorija(knjiga.prethodna), knjiga];
};

console.log("lanac unazad:", k3.prethodna === k2, k2.prethodna === k1, k1.prethodna === k0);
console.log("kraj lanca:", k0.prethodna);

console.log("\nrazvijeno u niz:", istorija(k3).length, "verzija");
console.log("stanje po verzijama:", istorija(k3).map((k) => k.stanje));
console.log("sadrzaj po verzijama:");
istorija(k3).forEach((v, i) => console.log("   " + i + ":", v.transakcije.map((t) => t.opis)));

---
# 6 · `pretraga` — sita jedno za drugim

## Slika

Naslagana sita. Sipaš sve odozgo. Ono što prođe prvo sito pada na drugo, pa na treće. Na kraju
ostane samo ono što je prošlo **sva**.

```js
const pretraga = function (kriterijumi, knjiga) {
  return kriterijumi.reduce((niz, kriterijum) => niz.filter(kriterijum), knjiga.transakcije);
};
```

`reduce` ovde ne sabira brojeve — **akumulator je niz transakcija**, a svaki korak ga propušta kroz
jedno sito. Početna vrednost je sve iz knjige.

Otud i „primena se vrši **onim redom kojim su navedeni**": redosled u nizu je redosled sita.

## Kriterijumi su obične funkcije

```js
const realizovane = (t) => t.status === "realizovana";
const preko = (granica) => (t) => t.iznos > granica;
```

`realizovane` je sito. `preko` **nije sito** — to je **fabrika sita**: `preko(50000)` tek pravi sito
sa tom rupom. Zato se poziva dvaput.

Nov kriterijum je jedan red i `pretraga` se ne dira.

## Kako se vidi da redosled zaista važi

Po **rezultatu** se ne vidi — `[A, B]` i `[B, A]` daju isti skup. Vidi se po **broju poziva**:
prvo sito vidi sve, drugo samo ono što je prošlo prvo.

In [ ]:
const pretraga = function (kriterijumi, knjiga) {
  return kriterijumi.reduce((niz, kriterijum) => niz.filter(kriterijum), knjiga.transakcije);
};

const realizovane = (t) => t.status === "realizovana";
const naTeret = (t) => t.tip === "na teret";
const preko = (granica) => (t) => t.iznos > granica;

console.log("jedno sito:", pretraga([realizovane], k3).map((t) => t.opis));
console.log("dva sita:  ", pretraga([realizovane, naTeret], k3).map((t) => t.opis));
console.log("fabrika sita:", pretraga([preko(50000)], k3).map((t) => t.opis));
console.log("bez sita:  ", pretraga([], k3).map((t) => t.opis));

// redosled se vidi po broju poziva, ne po rezultatu
const poziva = { A: 0, B: 0 };
const A = (t) => (poziva.A++, realizovane(t));
const B = (t) => (poziva.B++, naTeret(t));

const rezAB = pretraga([A, B], k3).map((t) => t.opis);
const redAB = [poziva.A, poziva.B];
poziva.A = 0; poziva.B = 0;
const rezBA = pretraga([B, A], k3).map((t) => t.opis);
const redBA = [poziva.A, poziva.B];

console.log("\n[A, B] pozivi:", redAB, "| [B, A] pozivi:", redBA);
console.log("prvo sito vidi sve, drugo samo preostalo");
console.log("rezultat je isti:", rezAB, rezBA);

---
# 7 · `opoziv` — storniranje

## Slika

Ovo je jedini deo gde je slika **bukvalno** knjigovodstvena praksa, a ne poređenje.

Kad se u knjizi napravi greška, ne briše se i ne precrtava. Unosi se **novi zapis** koji poništava
stari, i on se zove **storno**. Stari unos ostaje da stoji — vidljiv, sa svojim brojem — samo više
ne učestvuje u saldu.

Tačno to radi `opoziv`:

```js
const nastaleKasnije = verzije
  .filter((v, i) => i > redniBrojUIstoriji)               // sve posle zadate verzije
  .reduce((niz, v) => [...niz, ...v.transakcije], [])     // skupi njihove transakcije
  .filter((t) => brojeviStare.indexOf(t.redniBroj) === -1) // one kojih u zadatoj nije bilo
  .filter((t, i, niz) => niz.findIndex((x) => x.redniBroj === t.redniBroj) === i)  // bez ponavljanja
  .map(stornirana);                                        // KOPIJE sa statusom stornirana
```

## Tri stvari koje se lako promaše

**Zašto drugi `filter`.** Ista transakcija se nalazi u **više** kasnijih verzija — verzija 2 i verzija 3
obe sadrže `t2`. Bez izbacivanja ponavljanja, ista bi bila stornirana dvaput u istoj knjizi.

**Zašto `map(stornirana)`, a ne `t.status = "stornirana"`.** Zadatak izričito kaže da funkcija ne sme
da menja „nijednu od glavnih knjiga **niti neku od njihovih transakcija**". Prepisivanjem statusa
pokvario bi i sve **stare** verzije, jer one drže **iste** objekte — tačno ono što je pokazano u odeljcima 2 i 4.

```js
const stornirana = function (t) { return { ...t, status: "stornirana" }; };
```

Spread pravi **nov** objekat sa istim `redniBroj` — kao storno stavka koja se poziva na broj originala.

**Zašto se stanje poklapa.** Sve kasnije transakcije dobijaju status `stornirana`, a `stanje` sabira
**samo realizovane**. Ostaje tačno ono što je zadata verzija imala. Ta jednakost nije programirana —
ona je **posledica** definicije salda.

In [ ]:
const stornirana = function (t) {
  return { ...t, status: "stornirana" };
};

const opoziv = function (knjiga, redniBrojUIstoriji) {
  const verzije = istorija(knjiga);
  const stara = verzije[redniBrojUIstoriji];
  const brojeviStare = stara.transakcije.map((t) => t.redniBroj);

  const nastaleKasnije = verzije
    .filter((v, i) => i > redniBrojUIstoriji)
    .reduce((niz, v) => [...niz, ...v.transakcije], [])
    .filter((t) => brojeviStare.indexOf(t.redniBroj) === -1)
    .filter((t, i, niz) => niz.findIndex((x) => x.redniBroj === t.redniBroj) === i)
    .map(stornirana);

  return new GlavnaKnjiga(knjiga.naziv, knjiga.maticniBroj, knjiga.pib,
    [...stara.transakcije, ...nastaleKasnije], knjiga);
};

// zasto treba drugi filter: ista transakcija u vise verzija
const svePosle = istorija(k3)
  .filter((v, i) => i > 1)
  .reduce((niz, v) => [...niz, ...v.transakcije], []);
console.log("bez izbacivanja ponavljanja:", svePosle.map((t) => t.redniBroj), "— ponavlja se");

const op = opoziv(k3, 1);
console.log("\nverzija 1 je imala:", istorija(k3)[1].transakcije.map((t) => t.opis));
console.log("opozvana sadrzi:", op.transakcije.map((t) => t.opis + "(" + t.status + ")"));
console.log("stanje:", op.stanje, "= stanje verzije 1:", istorija(k3)[1].stanje);

console.log("\nnista nije prepravljeno:");
console.log("   originalne transakcije:", t2.status, "|", t3.status);
console.log("   tekuca knjiga:", k3.transakcije.map((t) => t.status));
console.log("   storno je KOPIJA:", op.transakcije[1] !== t2, "| isti redniBroj:", op.transakcije[1].redniBroj === t2.redniBroj);
console.log("\nistorija se nastavlja:", istorija(op).length, "verzija — i opoziv je dogadjaj u knjizi");

---
# Kako se sve spaja

Rešenje nije sedam nezavisnih delova nego jedan lanac:

1. **IIFE** drži brojač i spiskove van domašaja (1) → `redniBroj` je pouzdan identitet transakcije.
2. Pošto je identitet pouzdan, **`ukloniTransakciju`** ume da nađe pravu, a **`opoziv`** da prepozna
   koja se ponavlja (4 i 7).
3. **Geter sa kopijom** čuva spisak od spoljnih izmena (2) → nemutiranje ne zavisi od discipline pozivaoca.
4. **Metode vraćaju novu knjigu i pamte prethodnu** (4) → istorija nastaje sama (5).
5. Pošto stare verzije ostaju važeće, **`opoziv`** može da uzme bilo koju i sagradi novu (7).
6. **`stanje` se računa iz transakcija** (3) → zato se saldo opozvane poklapa sa starom verzijom,
   a da to niko nije posebno programirao.

## Tri rečenice za odbranu

> **Brojač je u IIFE, ne u konstruktoru.** IIFE se izvrši jednom, pa je knjiga jedna za ceo tip.
> Da je `let` bio u konstruktoru, svaki `new` bi otvorio novu knjigu i svi bi dobili broj 1.

> **Geter vraća kopiju, ne sam niz.** Bez toga bi `knjiga.transakcije.push(...)` menjao knjigu spolja,
> pa bi nemutiranje palo iako su metode ispravne.

> **`opoziv` ne prepravlja statuse nego pravi kopije.** Transakcije se **dele** između verzija,
> pa bi prepisivanje statusa pokvarilo i sve stare verzije, ne samo tekuću.